# Supplementary clinical PEB Modeling

For a complete rebuild: run the preparation cells below, run the existing peb_section4_final.m in MATLAB using that sample, then continue with the PEB reporting cells. If the current five PEB CSV exports were already generated from these same inputs, they can be read directly. MATLAB estimation remains in the existing script; it is not launched here.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# project PATH
DATASET_NAME = "sz_meditation"
project_path_str="/scratch/repo_"+DATASET_NAME
project_path= Path(project_path_str)
sys.path.append(str(project_path))

# importing private libs and functions (imaging_arsenal version, need to import the parent dir as well)
private_lib_path = Path('/scratch/imaging_arsenal')
sys.path.append(str(private_lib_path))
sys.path.append(str(private_lib_path / 'utils_'))
print('private lib ready:',(private_lib_path / 'utils_private.py').exists())
from utils_.utils_private import report_cpu, report_unified_missingness
report_cpu()

# data path
data_path = project_path / 'data' 
fig_path = project_path / 'fig' 

# data
basic_time_Tag = "20240724"
all_DKT_path = data_path / (DATASET_NAME+'_R_'+basic_time_Tag+'.csv')

# read data
all_df = pd.read_csv(all_DKT_path, index_col= None)

private lib ready: False


/opt/conda/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Total CPUs: 12
Total Memory: 29.38 GB
Available Memory: 22.78 GB


In [2]:
# report major software version
import statsmodels 
import nilearn
import sys
import sklearn
import semopy
import seaborn as sns
import networkx as nx
# setup random seed
rand_seed = 24
np.random.seed(rand_seed)

print('python version:', sys.version)
print('numpy version:', np.__version__)
print('statsmodels version:', statsmodels.__version__)
print('nilearn version:', nilearn.__version__)
print('sklearn version:', sklearn.__version__)
print('semopy version:', semopy.__version__)
print('networkX version:', nx.__version__)
print('seaborn version:', sns.__version__)
print('random seed:', rand_seed)

python version: 3.9.12 | packaged by conda-forge | (main, Mar 24 2022, 23:25:59) 
[GCC 10.3.0]
numpy version: 1.26.4
statsmodels version: 0.14.5
nilearn version: 0.12.0
sklearn version: 1.4.2
semopy version: 2.3.11
networkX version: 3.1
seaborn version: 0.13.2
random seed: 24


## 1. Read finalized Section 3 EC/motion data and original clinical records.

In [3]:
# Aim: rebuild the imaging-cohort inputs; method: participant/visit matching;
# output: raw EC, motion and clinical tables, independent of trajectory classes.
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from IPython.display import display

peb_path = data_path / 'dcm' / 'PEB_section3'
nc_path = data_path / 'neural_clinical'
nc_edges = pd.read_csv(peb_path / 'Selected_edge_dictionary.csv')
nc_brain = pd.read_csv(peb_path / 'DCM_selected_edges_all_available.csv',
                       dtype={'Participant_ID': str})
nc_brain = nc_brain.rename(columns={'Participant_ID': 'participant_id',
                                    'Session_Month': 'session', 'Group': 'group'})
nc_brain['session'] = pd.to_numeric(nc_brain['session'], errors='raise')
nc_brain['group'] = nc_brain['group'].str.strip().str.lower()
nc_discovery = nc_brain['In_PEB_Discovery'].astype(str).str.lower().isin(['1', '1.0', 'true'])
nc_brain = nc_brain.loc[nc_discovery & nc_brain.session.isin([0, 3, 8])].copy()
nc_ids = pd.Index(sorted(nc_brain.participant_id.unique()), name='participant_id')
nc_scores = ['PANSS_SUM', 'RBANS_sum', 'PANSS_P_SUM', 'PANSS_N_SUM', 'PANSS_G_SUM']
nc_baselines = ['PANSS_SUM_base', 'RBANS_sum_base']
nc_columns = ['participant_id', 'session', 'group', 'CPZ_dose_0'] + nc_scores + nc_baselines
nc_clinical = all_df[nc_columns].copy()
nc_clinical['participant_id'] = nc_clinical['participant_id'].astype(str)
nc_clinical['session'] = pd.to_numeric(nc_clinical['session'], errors='raise')
nc_clinical['group'] = nc_clinical['group'].str.strip().str.lower()
nc_clinical = nc_clinical.loc[nc_clinical.participant_id.isin(nc_ids) &
                              nc_clinical.session.isin([0, 3, 8])].copy()
nc_numeric = ['CPZ_dose_0'] + nc_scores + nc_baselines
nc_clinical[nc_numeric] = nc_clinical[nc_numeric].apply(pd.to_numeric, errors='raise')
nc_visits = {month: nc_clinical.loc[nc_clinical.session.eq(month)]
             .set_index('participant_id', verify_integrity=True).reindex(nc_ids)
             for month in [0, 3, 8]}
nc_brain_groups = nc_brain.pivot(index='participant_id', columns='session', values='group').reindex(nc_ids)
for month in [0, 3, 8]:
    present = nc_visits[month].group.notna() & nc_brain_groups[month].notna()
    if not nc_visits[month].loc[present, 'group'].eq(nc_brain_groups.loc[present, month]).all():
        raise ValueError('Clinical and DCM treatment labels disagree; resolve the source records before fitting.')

## 2. Define the common complete-case sample and rescale continuous inputs.


In [4]:
# Aim: reproduce the focused cohort; method: raw change definitions and observed ddof=0 SDs;
# output: one raw sample CSV for Python/MATLAB, plus in-memory standardized model inputs.
nc_raw = nc_visits[0][['group'] + nc_baselines].copy()
for score in nc_scores[2:]:
    nc_raw[score + '_base'] = nc_visits[0][score]
nc_raw['treatment'] = nc_raw.group.map({'control': 0, 'meditation': 1})
nc_raw['CPZ_dose'] = nc_visits[0]['CPZ_dose_0']
for month in [3, 8]:
    for score in (nc_scores if month == 3 else nc_scores[:2]):
        nc_raw[f'{score}_{month}'] = nc_visits[month][score]
    nc_raw[f'PANSS_dec_rate_{month}'] = (
        (nc_raw.PANSS_SUM_base - nc_raw[f'PANSS_SUM_{month}']) /
        (nc_raw.PANSS_SUM_base - 30).where(nc_raw.PANSS_SUM_base.gt(30)))
    nc_raw[f'RBANS_inc_rate_{month}'] = (
        (nc_raw[f'RBANS_sum_{month}'] - nc_raw.RBANS_sum_base) /
        nc_raw.RBANS_sum_base.where(nc_raw.RBANS_sum_base.gt(0)))
nc_fd = nc_brain.pivot(index='participant_id', columns='session', values='mean_fd').reindex(nc_ids)
for month in [0, 3, 8]:
    nc_raw[f'FD{month}'] = pd.to_numeric(nc_fd[month], errors='raise')
nc_focal_edges = nc_edges.loc[nc_edges.Source_Node.eq('IC02_SalVentA') &
                              nc_edges.Target_Node.isin(['IC00_SMN', 'IC18_SalVentB'])]
for edge in nc_focal_edges.itertuples(index=False):
    name = edge.Source_Node.split('_', 1)[1] + '_to_' + edge.Target_Node.split('_', 1)[1]
    ec = nc_brain.pivot(index='participant_id', columns='session', values=edge.CSV_Column).reindex(nc_ids)
    nc_raw['EC0_' + name] = ec[0]
    nc_raw['D3_' + name] = ec[3] - ec[0]
nc_model_columns = [
    'PANSS_dec_rate_3', 'RBANS_inc_rate_3', 'PANSS_dec_rate_8', 'RBANS_inc_rate_8',
    'treatment', 'PANSS_SUM_base', 'RBANS_sum_base', 'CPZ_dose',
    'EC0_SalVentA_to_SMN', 'EC0_SalVentA_to_SalVentB',
    'D3_SalVentA_to_SMN', 'D3_SalVentA_to_SalVentB', 'FD0', 'FD3', 'FD8',
]
nc_required = nc_raw[nc_model_columns].apply(pd.to_numeric, errors='raise').replace([np.inf, -np.inf], np.nan)
nc_missing = nc_required.isna()
print('Unavailable required values within the Section 3 discovery cohort:')
display(nc_missing.sum().loc[lambda x: x.gt(0)].rename('Missing_N').to_frame())
nc_sample = nc_raw.loc[~nc_missing.any(axis=1)].copy()
for score in ['PANSS_SUM', 'RBANS_sum']:
    for month in [0, 3, 8]:
        if not np.allclose(nc_visits[month].loc[nc_sample.index, score + '_base'],
                           nc_visits[0].loc[nc_sample.index, score], equal_nan=True):
            raise ValueError(f'{score}: baseline metadata differs across visits or from the actual baseline score.')
nc_continuous = [c for c in nc_model_columns if c != 'treatment']
nc_means = nc_sample[nc_continuous].mean()
nc_sds = nc_sample[nc_continuous].std(ddof=0)
fc_z = nc_sample[nc_model_columns].copy()
fc_z[nc_continuous] = (fc_z[nc_continuous] - nc_means) / nc_sds
fc_sd = fc_z.std(ddof=0)  # Fixed reference scales are retained during every omission.
nc_path.mkdir(parents=True, exist_ok=True)
nc_sample.to_csv(nc_path / 'Neural_clinical_sample.csv', index=True)
print(f'N={len(nc_sample)}; control={nc_sample.treatment.eq(0).sum()}; '
      f'meditation={nc_sample.treatment.eq(1).sum()}. No imputation.')
print('Two focal baseline ECs and their early changes.')

Unavailable required values within the Section 3 discovery cohort:


,Missing_N
RBANS_inc_rate_3,1
RBANS_inc_rate_8,1
RBANS_sum_base,1


N=43; control=21; meditation=22. No imputation.
Two focal baseline ECs and their early changes.


## MATLAB handoff

Preparation is complete. The following blocks require these five outputs from ```peb_section4_final.m``` in data/neural_clinical:

1. PEB_clinical_correspondence.csv;
2. PEB_clinical_conditioning.csv; 
3. PEB_clinical_raw_score.csv; 
4. PEB_clinical_temporal_blocks.csv;
5. PEB_clinical_LOO.csv


## 3. Read the complementary clinical-PEB tables

In [5]:
# Aim: define the reporting evidence; method: read existing MATLAB exports;
# output: unique clinical contrasts and a model/sample table, without refitting.
peb_files = {
    'Correspondence': 'PEB_clinical_correspondence.csv',
    'Conditioning': 'PEB_clinical_conditioning.csv',
    'Raw_score': 'PEB_clinical_raw_score.csv',
    'Temporal_blocks': 'PEB_clinical_temporal_blocks.csv',
    'Participant_omissions': 'PEB_clinical_LOO.csv',
}
peb_results = {name: pd.read_csv(nc_path / filename) for name, filename in peb_files.items()}
for name, table in peb_results.items():
    if not (table.EC_count.eq(8).all() and table.Edge_set.eq('Section3_original_eight').all()):
        raise ValueError(f'{name}: use the formal eight-edge export from peb_section4_final.m.')

peb_keys = ['Model', 'Clinical_regressor', 'Source_Node', 'Target_Node', 'EC_change_window']
peb_columns = peb_keys + [
    'Source_label', 'Target_label', 'FullPEB_Hz_per_clinical_SD', 'CrI95_low', 'CrI95_high',
    'N', 'Control_N', 'Meditation_N', 'EC_count', 'Edge_set',
    'Unit', 'Interval',
]
peb_all = pd.concat([peb_results[name][peb_columns] for name in
                     ['Correspondence', 'Conditioning', 'Raw_score']], ignore_index=True)
# Repeated copies of the same contrast occur in the three MATLAB reporting files.
peb_all = peb_all.drop_duplicates().set_index(peb_keys, verify_integrity=True).reset_index()
peb_all['Edge'] = (peb_all.Source_Node.str.split('_').str[0] + ' ' + peb_all.Source_label +
                   ' → ' + peb_all.Target_Node.str.split('_').str[0] + ' ' + peb_all.Target_label)
peb_status = peb_all[['Model', 'N', 'Control_N', 'Meditation_N', 'EC_count']].drop_duplicates()
print('Question: how do clinical responses correspond to neural trajectories, retaining DCM uncertainty?')
print('Clinical PEB models neural parameters; its slopes are not EC-to-clinical CLPM coefficients.')
display(peb_status)

Question: how do clinical responses correspond to neural trajectories, retaining DCM uncertainty?
Clinical PEB models neural parameters; its slopes are not EC-to-clinical CLPM coefficients.


,Model,N,Control_N,Meditation_N,EC_count
0,Early_correspondence,43,21,22,8
16,Late_conditional_correspondence,43,21,22,8
32,PDR3_only,43,21,22,8
40,RIR3_only,43,21,22,8
48,Raw_score_early,43,21,22,8


## 4. Focal correspondence and saved participant influence

In [6]:
# Aim: describe IC02-to-IC00 symptom correspondence; method: fixed edge/contrast lookup;
# output: pointwise credible intervals and upper-model omission ranges, with no significance filtering.
peb_influence = peb_results['Participant_omissions']
peb_loo_columns = [
    'LOO_usable_min', 'LOO_usable_max', 'LOO_usable_opposite_sign_count',
    'Usable_omissions', 'Failed_omissions', 'Not_run_omissions',
    'Total_planned_omissions',
]
peb_reference = peb_influence.loc[peb_influence.Fit_status.eq('Reference'),
                                 peb_keys + peb_loo_columns]
peb_all = peb_all.drop(columns=peb_loo_columns + ['Review_omissions'], errors='ignore').merge(
    peb_reference, on=peb_keys, how='left', validate='one_to_one')
peb_focal = peb_all.loc[peb_all.Source_Node.eq('IC02_SalVentA') &
                       peb_all.Target_Node.eq('IC00_SMN')].copy()
peb_early = peb_focal.loc[peb_focal.Clinical_regressor.eq('PANSS_dec_rate_3') &
                         peb_focal.Model.isin(['Early_correspondence', 'PDR3_only'])].copy()
peb_early['Clinical_conditioning'] = peb_early.Model.map({
    'Early_correspondence': 'PDR3 with RIR3 adjustment',
    'PDR3_only': 'PDR3 without RIR3 adjustment',
})
print('IC02 SMG–VAN → IC00 SMN: early proportional symptom improvement and 0–3-month EC change.')
print('This is a result-informed reporting focus within the original eight-edge analysis.')
display(peb_early[[
    'Clinical_conditioning', 'FullPEB_Hz_per_clinical_SD', 'CrI95_low', 'CrI95_high',
]].round(4))
display(peb_early.loc[peb_early.Model.eq('Early_correspondence'),
                      ['Model'] + peb_loo_columns].round(4))

# Keep the saved omission summaries unchanged; do not relabel historical fit records.
peb_failed = peb_influence.loc[peb_influence.Fit_status.eq('Failed'),
                                ['Omitted_ID', 'Error']].drop_duplicates()
if not peb_failed.empty:
    print('Saved omission errors:')
    display(peb_failed)
print('Omission ranges use the records included in the saved summary; no statuses are reclassified.')

print('Positive slope: greater proportional symptom improvement corresponds to more positive EC change.')
print('Intervals are pointwise 95% credible intervals, not adjusted for the reporting selection.')
print('Omissions refit the upper model with the remaining participant-level PEBs and fixed full-sample scales. '
      'Ranges describe influence, not credible intervals or independent replication.')
print('Both quantities cover 0–3 months. The neural trajectory uses all three visits and FD0/FD3/FD8; '
      'this is retrospective correspondence, not prospective prediction, mediation or meditation-specific coupling.')

IC02 SMG–VAN → IC00 SMN: early proportional symptom improvement and 0–3-month EC change.
This is a result-informed reporting focus within the original eight-edge analysis.


,Clinical_conditioning,FullPEB_Hz_per_clinical_SD,CrI95_low,CrI95_high
0,PDR3 with RIR3 adjustment,0.3056,0.0909,0.5203
32,PDR3 without RIR3 adjustment,0.2636,0.0489,0.4784


,Model,LOO_usable_min,LOO_usable_max,LOO_usable_opposite_sign_count,Usable_omissions,Failed_omissions,Not_run_omissions,Total_planned_omissions
0,Early_correspondence,0.242,0.3509,0.0,43.0,0.0,0.0,43.0


Omission ranges use the records included in the saved summary; no statuses are reclassified.
Positive slope: greater proportional symptom improvement corresponds to more positive EC change.
Intervals are pointwise 95% credible intervals, not adjusted for the reporting selection.
Omissions refit the upper model with the remaining participant-level PEBs and fixed full-sample scales. Ranges describe influence, not credible intervals or independent replication.
Both quantities cover 0–3 months. The neural trajectory uses all three visits and FD0/FD3/FD8; this is retrospective correspondence, not prospective prediction, mediation or meditation-specific coupling.


## 5. Clinical representation and temporal scope

In [7]:
# Aim: bound the focal interpretation; method: display existing related contrasts and block evidence;
# output: alternative clinical questions and their estimates, without requiring agreement across definitions.
peb_context = peb_focal.loc[
    (peb_focal.Model.eq('Raw_score_early') & peb_focal.Clinical_regressor.eq('PANSS3_lower_z')) |
    (peb_focal.Model.eq('Late_conditional_correspondence') &
     peb_focal.Clinical_regressor.eq('PANSS_dec_rate_8'))
].copy()
peb_context['Question'] = peb_context.Model.map({
    'Raw_score_early': 'Early EC change and lower month-3 PANSS, conditional on baseline',
    'Late_conditional_correspondence': 'Cumulative EC change and PDR8, conditional on early responses',
})
display(peb_context[[
    'Question', 'Clinical_regressor', 'EC_change_window', 'FullPEB_Hz_per_clinical_SD',
    'CrI95_low', 'CrI95_high',
]].round(4))
print('The raw-score specification adjusts for baseline PANSS/RBANS and contemporaneous RBANS3.')

peb_blocks = peb_results['Temporal_blocks'].copy()
display(peb_blocks[[
    'Clinical_definition', 'Model', 'Temporal_coefficients_per_clinical_block',
    'LogBF_vs_both_temporal', 'Posterior_model_probability',
]].round(6))
print('Block evidence compares four models within each clinical definition. Each clinical temporal block '
      'contains linear and quadratic terms for all eight ECs; visit-average clinical and nuisance effects remain free.')
print('These are broader hypotheses than the focal contrast. Probabilities depend on this model family and its priors; '
      'they are not probabilities that brain–clinical biology is present or absent. (Do not compare log Bayes factors across definitions !)')

,Question,Clinical_regressor,EC_change_window,FullPEB_Hz_per_clinical_SD,CrI95_low,CrI95_high
16,"Cumulative EC change and PDR8, conditional on ...",PANSS_dec_rate_8,0-8 months,-0.0887,-0.3292,0.1518
48,"Early EC change and lower month-3 PANSS, condi...",PANSS3_lower_z,0-3 months,0.1593,-0.2057,0.5242


The raw-score specification adjusts for baseline PANSS/RBANS and contemporaneous RBANS3.


,Clinical_definition,Model,Temporal_coefficients_per_clinical_block,LogBF_vs_both_temporal,Posterior_model_probability
0,Ratio,Neither_temporal,16,14.283359,0.999305
1,Ratio,Symptoms_temporal,16,6.046007,0.000264
2,Ratio,Cognition_temporal,16,6.532531,0.000430
3,Ratio,Both_temporal,16,0.000000,0.000001
4,Raw_score,Neither_temporal,16,7.770181,0.944584
5,Raw_score,Symptoms_temporal,16,2.744849,0.006205
6,Raw_score,Cognition_temporal,16,4.807401,0.048811
7,Raw_score,Both_temporal,16,0.000000,0.000399


Block evidence compares four models within each clinical definition. Each clinical temporal block contains linear and quadratic terms for all eight ECs; visit-average clinical and nuisance effects remain free.
These are broader hypotheses than the focal contrast. Probabilities depend on this model family and its priors; they are not probabilities that brain–clinical biology is present or absent. (Do not compare log Bayes factors across definitions !)


## 6. Supplementary reporting tables

In [8]:
# Aim: prepare concise supplementary reporting; method: retain all fitted clinical contrasts;
# output: two optional CSV tables, without new fits, figures, p-values or additional selection.
save_peb_tables = 0
show_all_peb_contrasts = 0
peb_sm = peb_all.copy()
peb_sm['Reporting_focus'] = (
    peb_sm.Source_Node.eq('IC02_SalVentA') & peb_sm.Target_Node.eq('IC00_SMN') &
    peb_sm.Model.eq('Early_correspondence') & peb_sm.Clinical_regressor.eq('PANSS_dec_rate_3'))
peb_sm['Reporting_scope'] = 'Exploratory, result-informed focus; complete fitted contrast family retained'
peb_focal_report = peb_sm.loc[
    peb_sm.Source_Node.eq('IC02_SalVentA') & peb_sm.Target_Node.eq('IC00_SMN') &
    peb_sm.Clinical_regressor.isin(['PANSS_dec_rate_3', 'PANSS_dec_rate_8', 'PANSS3_lower_z'])
].copy()
if show_all_peb_contrasts:
    with pd.option_context('display.max_rows', None):
        display(peb_sm[['Model', 'Clinical_regressor', 'EC_change_window', 'Edge',
                        'FullPEB_Hz_per_clinical_SD', 'CrI95_low', 'CrI95_high']].round(4))
if save_peb_tables:
    peb_focal_report.to_csv(nc_path / 'PEB_IC02_to_IC00_reporting.csv', index=False)
    peb_sm.to_csv(nc_path / 'PEB_clinical_complete_contrasts_SM.csv', index=False)
    print('Saved the focal reporting table and complete clinical-PEB contrast table in:', nc_path)
print('The existing PEB_clinical_temporal_blocks.csv accompanies these tables; it is not duplicated.')
print('Interpretation: assess the focal symptom correspondence alongside its clinical-conditioning, '
      'participant-influence and temporal-block results.')
print('This PEB result does not validate the CLPM baseline-EC/cognition association, establish an EC-change '
      'mediator, or quantify additional variance explained in clinical outcomes.')

The existing PEB_clinical_temporal_blocks.csv accompanies these tables; it is not duplicated.
Interpretation: assess the focal symptom correspondence alongside its clinical-conditioning, participant-influence and temporal-block results.
This PEB result does not validate the CLPM baseline-EC/cognition association, establish an EC-change mediator, or quantify additional variance explained in clinical outcomes.
